# Generating Company Names with a Character-Level Transformer

**Author:** Mauro Reverberi

**Program:** MSc AI, Udacity Institute of AI & Technology / Woolf

**Project:** Capstone Project 5, Generative AI Applications

**Dataset:** Companies House, Free Company Data Product, snapshot of 2026-10-01:
https://download.companieshouse.gov.uk/en_output.html

TBD

**Research question:** Can a small character-level Transformer generate plausible and varied company names without mostly copying the names it was trained on or the names of companies that already exist?

I picked this task because it connects to my earlier capstone projects, which all work with company data for compliance checks. Project 1 built a dataset of Swiss legal entities from the GLEIF register, and Projects 2 to 4 analyzed company registrations, a bankruptcy score and negative news about companies. A later capstone project will build a due diligence agent that matches company names against registers and lists. Generated names could serve as extra test inputs for this step, names that look real but should not match anything. They are not a matching benchmark, which would need pairs of names that are known to be the same or different companies.

## Problem definition and dataset

**Task type:** TBD

**Dataset:** Companies House publishes a free monthly snapshot of all live companies on the UK register. I downloaded the snapshot of 2026-10-01 with 5,704,711 companies. Companies House imposes no rules on how the information on the public register is used, but rights of third parties in the data remain. The names are real register entries, nothing is synthetic, and I did not use this source in my earlier capstone projects.

`prepare_dataset.py` keeps the private limited companies whose names use only capital letters, digits and common punctuation, have at most 60 characters and end with LTD, LTD. or LIMITED. From the 5,259,281 different names it draws a random sample of 100,000 with seed 42. Very similar names like ACME HOLDINGS LTD and ACME HOLDINGS (UK) LIMITED form one group, so they cannot end up in training and test. The script splits the groups about 80/10/10 into train, validation and test. Names with the same first word and a different second word can still land in different splits.

The script also computes a key for every live company. The key follows the "same as" rules that Companies House uses to refuse a new name (Schedule 3 of the Names and Trading Disclosures Regulations 2015), so HANDS UK LTD and HAND'S LIMITED get the same key, and so do ACME 2 LTD and ACME TWO LTD. I wrote the key myself, and it is not the official test. It does not cover the exceptions or the separate "too like" test, which Companies House decides case by case. So a key match is only a possible collision with a real company, and a name without a match can still be too close to one. The 64-bit hashes of the keys are in `register_key_hashes.npy`, so a name can be checked against the register without keeping 5.7 million names in the repository.

## Setup

I use CUDA if it is available, on my Mac the Apple GPU (MPS), otherwise the CPU. Seed 42 is the main seed of the notebook, the same as in `prepare_dataset.py`.

In [1]:
import os
import random

import numpy as np
import pandas as pd
import seaborn as sns
import torch

from constants import DATA_FILE, REGISTER_FILE, SEED

In [2]:
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)
# deterministic algorithms where possible, CUDA needs the cuBLAS setting, warn_only avoids errors
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
torch.use_deterministic_algorithms(True, warn_only=True)

In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"device: {device}")

device: mps


In [4]:
def set_seed(seed):
    """Set the seed of random, NumPy and PyTorch, so a run can be repeated."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

## Load the dataset

I load the prepared sample, one row per company name, and the hashes of the register keys. I check the structure before I touch anything.

In [5]:
names = pd.read_csv(DATA_FILE, keep_default_na=False)
names["split"] = pd.Categorical(names["split"], ["train", "validation", "test"])
register_keys = np.load(REGISTER_FILE)
print(f"Rows: {names.shape[0]:,}, columns: {names.shape[1]}")
print(f"Register keys: {len(register_keys):,}")

Rows: 100,000, columns: 3
Register keys: 5,673,818


In [6]:
names.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype   
---  ------  --------------   -----   
 0   name    100000 non-null  str     
 1   group   100000 non-null  int64   
 2   split   100000 non-null  category
dtypes: category(1), int64(1), str(1)
memory usage: 1.6 MB


In [7]:
names.head()

,name,group,split
0,CF TROY UB INSURED LIMITED,17178,train
1,AHQENZA LIMITED,3582,train
2,INNSIGHT ADVISORS LTD,42646,train
3,HANN ELECTRICAL LTD,37927,train
4,GG-030-615 LIMITED,34636,train


The file has 100,000 rows and three columns, the name, the number of its name group and the split. No value is missing. All names are in capital letters, because the register stores almost all names this way and `prepare_dataset.py` removes the few others. Next to ordinary names like HANN ELECTRICAL LTD the first rows already show GG-030-615 LIMITED, which looks more like a code than a name.

The register file has 5,673,818 different keys for the 5,704,711 live companies of the snapshot, 30,893 fewer keys than company records. Some names may be allowed within a group of companies or with consent, and my key may also merge names that the official rules keep apart. I did not count the records per key.